In [1]:
None

In [2]:
import os
import os.path as osp
import json
import re
from tqdm import tqdm
from copy import deepcopy

def load_json(path):
    with open(path, 'r') as jf:
        data = json.load(jf)
    return data

def dump_json(obj, path, indent=2):
    with open(path, 'w') as jf:
        json.dump(obj, jf, indent=indent, ensure_ascii=False)

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            data.append(json.loads(line))
    return data

def dump_jsonl(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        for item in data:
            json_line = json.dumps(item, ensure_ascii=False)
            file.write(json_line + '\n')

def bboxes_to_str(bboxes, ostart='<bbox_list>', oend='</bbox_list>', instart='<box>', inend='</box>'):
    return ostart + "".join(
        f"{instart}{x1}, {y1}, {x2}, {y2}{inend}" for x1, y1, x2, y2 in bboxes
    ) + oend

def str_to_bboxes(s, ostart='<bbox_list>', oend='</bbox_list>', instart='<box>', inend='</box>'):
    blocks = re.findall(re.escape(ostart) + r"(.*?)" + re.escape(oend), s, re.S)
    if len(blocks) != 1:
        raise ValueError(f"Not 1 but {len(blocks)} blocks found")
    return [
        [int(v) for v in m.split(",")]
        for m in re.findall(
            re.escape(instart) + r"(.*?)" + re.escape(inend),
            blocks[0]
        )
    ]

def decimal2int(input_string, start_token='<bbox_list>', end_token='</bbox_list>'):
    pattern = re.compile(re.escape(start_token) + r'(.*?)' + re.escape(end_token), re.DOTALL)
    
    def convert(match):
        inner_text = match.group(1)
        # 匹配所有正负小数和整数
        transformed = re.sub(
            r'-?\d+\.?\d*',
            lambda x: str(min(1000, max(0, round(float(x.group()) * 1000)))),
            inner_text
        )
        return f"{start_token}{transformed}{end_token}"
    
    return re.sub(pattern, convert, input_string)

In [3]:
import os
import base64
import asyncio
from typing import List, Dict, Any, Union
from openai import OpenAI, AsyncOpenAI, APITimeoutError, APIConnectionError
from tenacity import (
    retry, 
    stop_after_attempt, 
    wait_random_exponential, 
    retry_if_exception_type
)

class MultiModalLLM:
    def __init__(self, provider="gemini", model=None, timeout=60.0, max_tokens=None):
        """
        :param provider: 'openai', 'gemini', 'deepseek'
        :param timeout: 全局超时时间（秒）
        """
        self.provider = provider.lower()
        self.timeout = timeout
        self.max_tokens = max_tokens
        
        # 配置各家厂商
        configs = {
            "openai": {
                "api_key": os.environ.get("OPENAI_API_KEY"),
                "base_url": None,
                "model": model or "gpt-5.2"
            },
            "gemini": {
                "api_key": os.environ.get("GEMINI_API_KEY"),
                "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
                "model": model or "gemini-3-flash-preview"
            },
            "deepseek": {
                "api_key": os.environ.get("DEEPSEEK_API_KEY"),
                "base_url": "https://api.deepseek.com",
                "model": model or "deepseek-chat"
            }
        }

        config = configs.get(self.provider)
        if not config or not config["api_key"]:
            raise ValueError(f"Provider {self.provider} 配置无效或缺少 API Key")

        self.model_name = config["model"]
        
        # 1. 同步客户端 (用于 chat_with_image)
        self.sync_client = OpenAI(
            api_key=config["api_key"],
            base_url=config["base_url"],
            timeout=self.timeout
        )
        
        # 2. 异步客户端 (用于 run_batch)
        self.async_client = AsyncOpenAI(
            api_key=config["api_key"],
            base_url=config["base_url"],
            timeout=self.timeout
        )

    def _encode_image(self, image_path):
        """读取图片并转Base64"""
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    def _prepare_payload(self, prompt, image_path, detail, json_mode):
        """内部工具：构造请求参数"""
        # 1. 处理 JSON 提示
        final_prompt = prompt
        if json_mode and "json" not in prompt.lower():
            final_prompt += " (Please output in JSON format)"

        # 2. 构造消息体
        messages = [{"role": "user", "content": [{"type": "text", "text": final_prompt}]}]
        if image_path:
            base64_image = self._encode_image(image_path)
            messages[0]["content"].append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}",
                    "detail": detail
                }
            })
        
        # 3. 构造参数字典
        kwargs = {
            "model": self.model_name,
            "messages": messages,
            "max_tokens": self.max_tokens,
        }
        
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
            
        return kwargs

    @retry(
        retry=retry_if_exception_type((APITimeoutError, APIConnectionError)),
        wait=wait_random_exponential(min=1, max=10),
        stop=stop_after_attempt(3),
        reraise=True 
    )
    def chat_with_image(self, prompt: str, image_path: str = None, detail: str = "auto", json_mode: bool = False) -> str:
        """
        同步方法：直接调用，阻塞直到返回结果。
        无需 await，兼容旧代码。
        """
        kwargs = self._prepare_payload(prompt, image_path, detail, json_mode)
        
        # 使用 sync_client
        response = self.sync_client.chat.completions.create(**kwargs)
        return response.choices[0].message.content

    @retry(
        retry=retry_if_exception_type((APITimeoutError, APIConnectionError)),
        wait=wait_random_exponential(min=1, max=10),
        stop=stop_after_attempt(3),
        reraise=True 
    )
    async def _chat_with_image_async(self, prompt: str, image_path: str = None, detail: str = "auto", json_mode: bool = False) -> str:
        """内部异步方法，用于并发"""
        kwargs = self._prepare_payload(prompt, image_path, detail, json_mode)
        
        # 使用 async_client
        response = await self.async_client.chat.completions.create(**kwargs)
        return response.choices[0].message.content

    async def run_batch(self, inputs: List[Dict[str, Any]], concurrency_limit: int = 5, json_mode: bool = False) -> List[Union[str, Dict]]:
        """
        异步批量处理。
        :param inputs: 参数列表
        :param concurrency_limit: 并发限制
        :param json_mode: 是否统一开启 JSON 模式 (如果不传，会优先看 inputs 里的 individual json_mode)
        """
        semaphore = asyncio.Semaphore(concurrency_limit)

        async def _safe_task(task_params):
            async with semaphore:
                try:
                    # 允许 inputs 中的字典单独覆盖 json_mode，否则使用全局传入的 json_mode
                    local_json_mode = task_params.get("json_mode", json_mode)
                    # 显式提取 key 避免传参报错
                    p = task_params.get("prompt")
                    img = task_params.get("image_path")
                    dtl = task_params.get("detail", "auto")
                    
                    return await self._chat_with_image_async(p, img, dtl, local_json_mode)
                except Exception as e:
                    return f"Error: {str(e)}"

        tasks = [_safe_task(params) for params in inputs]
        results = await asyncio.gather(*tasks)
        return list(results)

# # Test code
# client = MultiModalLLM('gemini')
# print('-----------------')
# print(client.chat_with_image('简短描述图片内容', '/c22073/datasets/RSN_LOC/MVI/pos/images/tumor cell cluster/1M05_4.jpg'))
# print('-----------------')
# print(client.chat_with_image('简短描述图片内容', '/c22073/datasets/RSN_LOC/MVI/pos/images/tumor cell cluster/1M05_4.jpg', json_mode=True))

# tasks = [
#     {"prompt": "简短描述图片内容", "image_path": '/c22073/datasets/RSN_LOC/MVI/pos/images/tumor cell cluster/1M05_4.jpg'},
#     {"prompt": "简短描述图片内容", "image_path": '/c22073/datasets/RSN_LOC/MVI/pos/images/tumor cell cluster/1M05_4.jpg', "json_mode": True},
#     {"prompt": "简短描述图片内容"} 
# ]
# results = await client.run_batch(tasks, concurrency_limit=5)
# for i, r in enumerate(results):
#     print(f'---------{i}--------')
#     print(r)

# 1 利用现有检测分割数据构造新数据

## 1.1 用MVI验证数据构造方案

### 1.1.1 生成元数据

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image
import re
import torch

def bboxes_to_str(bboxes, ostart='<bbox_list>', oend='</bbox_list>', instart='<box>', inend='</box>'):
    return ostart + "".join(
        f"{instart}{x1}, {y1}, {x2}, {y2}{inend}" for x1, y1, x2, y2 in bboxes
    ) + oend

def str_to_bboxes(s, ostart='<bbox_list>', oend='</bbox_list>', instart='<box>', inend='</box>'):
    blocks = re.findall(re.escape(ostart) + r"(.*?)" + re.escape(oend), s, re.S)
    if len(blocks) == 0:
        return ""
    if len(blocks) > 1:
        raise ValueError("multiple bbox_list blocks found")
    return [
        [int(v) for v in m.split(",")]
        for m in re.findall(
            re.escape(instart) + r"(.*?)" + re.escape(inend),
            blocks[0]
        )
    ]

def build_mvi_json(image_dir: str, labels_dir: str):
    """
    遍历 labels_dir 下所有子类文件夹与其 txt 标注（YOLO格式，类别恒为0），
    在 image_dir 对应子类文件夹中查找同名图片，生成列表。
    """
    image_dir = Path(image_dir)
    labels_dir = Path(labels_dir)

    img_exts = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]

    def find_image(cls_dir: Path, stem: str) -> Path:
        # 优先直接按常见后缀尝试，其次兜底遍历
        for ext in img_exts:
            p = cls_dir / f"{stem}{ext}"
            if p.exists():
                return p
        for p in cls_dir.iterdir():
            if p.is_file() and p.suffix.lower() in img_exts and p.stem == stem:
                return p
        raise FileNotFoundError(f"找不到图片：class_dir={cls_dir}, name_stem={stem}")

    def clamp_0_1000(v: int) -> int:
        return 0 if v < 0 else (1000 if v > 1000 else v)

    def yolo_to_bbox_0_1000(line: str, W: int, H: int):
        # YOLO: cls xc yc w h（均为0-1）
        parts = line.strip().split()
        if len(parts) < 5:
            return None
        _, xc, yc, bw, bh = parts[:5]
        xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)

        x1 = (xc - bw / 2) * W
        y1 = (yc - bh / 2) * H
        x2 = (xc + bw / 2) * W
        y2 = (yc + bh / 2) * H

        # 归一化到0-1000整数（相对各自维度）
        x1n = clamp_0_1000(int(round(x1 / W * 1000)))
        y1n = clamp_0_1000(int(round(y1 / H * 1000)))
        x2n = clamp_0_1000(int(round(x2 / W * 1000)))
        y2n = clamp_0_1000(int(round(y2 / H * 1000)))
        return [x1n, y1n, x2n, y2n]

    items = []

    # 以 labels_dir 为准遍历
    for cls_dir in sorted([p for p in labels_dir.iterdir() if p.is_dir()]):
        cls_name = cls_dir.name
        img_cls_dir = image_dir / cls_name
        if not img_cls_dir.exists():
            raise FileNotFoundError(f"图像子文件夹不存在：{img_cls_dir}")

        for txt_path in sorted(cls_dir.rglob("*.txt")):
            stem = txt_path.stem
            img_path = find_image(img_cls_dir, stem)

            with Image.open(img_path) as im:
                W, H = im.size

            bboxes = []
            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    box = yolo_to_bbox_0_1000(line, W, H)
                    if box is not None:
                        bboxes.append(box)
            bboxes = sorted(bboxes, key=lambda x: (x[0], x[1]))

            items.append({
                "image_path": str(img_path.resolve()),
                "width": int(W),
                "height": int(H),
                "metadata": {
                    "organ": "liver",
                    "stain": "H&E",
                    "diagnosis": "MVI positive",
                    "visual_evidence": {
                        cls_name: bboxes_to_str(bboxes)
                    }
                }
            })

    return items

In [ ]:
meta_data = build_mvi_json('/c22073/datasets/RSN_LOC/MVI/pos/images', '/c22073/datasets/RSN_LOC/MVI/pos/labels')
len(meta_data)

In [ ]:
train_convs = load_json('/c22073/datasets/VLM_MVI/set1/mvi_cancerous_nucleus/train_convs.json')
train_img_names = [osp.basename(x['image']) for x in train_convs]
print(len(train_img_names))

In [ ]:
# 删除不在训练集中的元数据
meta_data = [x for x in meta_data if osp.basename(x['image_path']) in train_img_names]
print(len(meta_data))

In [ ]:
# 补齐癌细胞核的数据
img_dir = '/c22073/datasets/RSN_LOC/MVI/pos/images/tumor cell nucleus'
origin_label_dir = '/c22073/datasets/VLM_MVI/set1/mvi_cancerous_nucleus/origin_labels'
olarge_img_dir = '/c22073/codes/swift-new/datasets/VLM_MVI/set1/mvi_cancerous_nucleus/origin_images/large'

def get_bbox_from_origin_label(oriimg, orilabels):
    import cv2
    oriimg = cv2.imread(oriimg)
    orilabels = torch.load(orilabels)
    pos_bboxes = [cv2.boundingRect(x) for x in orilabels['positive_cells']]
    assert oriimg.ndim == 3 and oriimg.shape[-1] == 3, oriimg.shape
    oh, ow = oriimg.shape[:2]
    if oh != ow:
        max_hw = max(oh, ow)
        if oh > ow:
            pad_idx = (oh - ow) // 2
            pos_bboxes = [(x+pad_idx, y, w, h) for x, y, w, h in pos_bboxes]
        else:
            pad_idx = (ow - oh) // 2
            pos_bboxes = [(x, y+pad_idx, w, h) for x, y, w, h in pos_bboxes]
    else:
        max_hw = oh
    edge = max_hw
    bbox_list = [[round(x/edge*1000), round(y/edge*1000), round((x+w)/edge*1000), round((y+h)/edge*1000)] for x, y, w, h in pos_bboxes]
    bbox_list.sort(key=lambda x: (x[0], x[1]))
    return bbox_list

for img in sorted(os.listdir(img_dir)):
    if img not in train_img_names:
        continue
    conv = [x for x in train_convs if osp.basename(x['image']) == img]
    assert len(conv) == 1, img
    conv, oriimg = conv[0]['conversations'], conv[0]['image']
    bboxes = None
    for c in conv:
        if c['from'] == 'gpt' and '<bbox_list>' in c['value']:
            bboxes = bboxes_to_str(str_to_bboxes(decimal2int(c['value']).replace('<bbox>', '<box>').replace('</bbox>', '</box>')))
            break
    else:
        assert 'large' in oriimg
        bboxes = get_bbox_from_origin_label(osp.join(olarge_img_dir, img), osp.join(origin_label_dir, osp.splitext(img)[0]+'_annots.pth'))
        bboxes = bboxes_to_str(bboxes)
    
    assert isinstance(bboxes, str)

    img_path = osp.join(img_dir, img)
    with Image.open(img_path) as im:
        W, H = im.size
    
    meta_data.append({
        "image_path": img_path,
        "width": int(W),
        "height": int(H),
        "metadata": {
            "organ": "liver",
            "stain": "H&E",
            "diagnosis": "MVI positive",
            "visual_evidence": {
                osp.basename(img_dir): bboxes
            }
        }
    })

print(len(meta_data))

In [ ]:
meta_data

In [ ]:
dump_json(meta_data, '/c22073/datasets/RSN_LOC/MVI/pos/train_metadata.json')

In [ ]:
# 可视化视觉证据看看准不准
import os
import re
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

def visualize_visual_evidence(meta_list, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 16)
    except Exception:
        font = ImageFont.load_default()

    for i, item in enumerate(meta_list):
        img_path = Path(item["image_path"])
        ve = (item.get("metadata") or {}).get("visual_evidence") or {}
        if not ve:
            continue

        im = Image.open(img_path).convert("RGB")
        W, H = im.size
        draw = ImageDraw.Draw(im)

        for cls_name, ve_val in ve.items():
            bboxes = str_to_bboxes(ve_val)
            for x1, y1, x2, y2 in bboxes:
                x1 = int(round(x1 / 1000 * W)); y1 = int(round(y1 / 1000 * H))
                x2 = int(round(x2 / 1000 * W)); y2 = int(round(y2 / 1000 * H))
                draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
                tw, th = draw.textbbox((0, 0), cls_name, font=font)[2:]
                tx, ty = x1, max(0, y1 - th - 4)
                draw.rectangle([tx, ty, tx + tw + 6, ty + th + 4], fill="red")
                draw.text((tx + 3, ty + 2), cls_name, fill="white", font=font)

        im.save(out_dir / img_path.name)
        im.close()

In [ ]:
visualize_visual_evidence(meta_data, '/c22073/datasets/RSN_LOC/MVI/pos/train_metadata_visualize')

### 1.1.2 调用API生成数据

In [ ]:
from tqdm import tqdm

In [ ]:
BASE_PROMPT = """
你是一名资深病理专家，负责出具病理医生资格考试题目。我将为你提供一张病理图片，和与其对应的json格式元数据。

你需要根据病理图像、元数据和任务要求，完成如下两个任务，并将两个任务的结果合并为一个json字典输出。

**任务1：**

{task1_requirement}

**任务2：**

{task2_requirement}

元数据是给你提供信息参考所用，在构造数据的任何部分不要提及元数据的存在。

构造的内容的语言和本Prompt语言相同。请按照要求返回一个能被直接解析的json字典字符串，不需要放在代码块里，前后不要有任何其他内容。

元数据：{meta_data}
""".strip()

DIAG_RELATED_QUENSTION_GENERATE_PROMPT = """
你需要根据图像实际内容，和元数据中的信息，构造和元数据中“diagnosis”相关的单项选择题。要求：
- 选项数不少于3个
- 问题**必须**通过分析图像才能正确回答，禁止构造仅凭文本问题就可以直接选择正确答案的问答对
- 元数据中的信息是真实的，生成问题时可以参考。
- 问题中可出现一些元数据中的基础信息，如“organ”等。但是和问答相关的关键信息，如“diagnosis”、“visual_evidence”等不能出现在问题中。
- 返回的json字典中添加question字段存储问题字符串，options存储选项字符串列表，gt存储正确答案在选项列表中的index，visual_integration_challenges存储为什么这个问题一定要基于图像才能正确回答
- options中的字符串仅提供选项内容，不要有选项序号
- 正确答案的位置放置在随机位置
""".strip()

RSN_WITH_ENTITY_PROMPT = """
你需要对构造的问题和回答生成标准答案解析。遵循如下基本格式：
在给出回答之前，**必须**先在心里进行System 2式思考，将思考内容放在<think>和</think>之间，在</think>之后进行回答。对于有确定性答案的问题，如选择题和填空题等，在回答的最后，将选项或答案放在<answer>和</answer>之间。例如：<think>思考内容</think>思考完成后的回答内容和分析<answer>选项或回答短语</answer>

此外，为了让回答有可信依据，需要遵循如下要求：

----------- 要求开始 -----------
在分析过程中，当提及图像中的关键实体证据（如细胞、组织结构、区域）时，请严格遵守以下 XML 格式规范，使用<entity>与<ref>标签来绑定“关键实体证据”：

1）首次提及可见实体（阳性）：
- 当某个概念/名词作为关键证据第一次出现时，必须用<entity>标签直接替代该名词进行定义。
- 确保该实体在图中出现，需在标签内给出对其的一个或多个检测框。
- 标准格式：<entity name="实体名" id="唯一整数ID"><bbox_list><box>x1, y1, x2, y2</box></bbox_list></entity>
- 在对话历史中，实体id必须唯一，且为从1递增的整数。

2）首次提及阴性证据（图中未出现）：
- 当“某实体未出现”本身是关键证据时，必须用自闭合标签定义：<entity name="实体名" id="唯一整数ID" status="neg"/>
- 阴性实体不提供检测结果，但要设置属性status为"neg"。

3）后续再次提及已定义的实体，需准确引用：
- 必须使用自闭合标签，标准格式：<ref name="实体名" id="同一个ID"/>
- 不得重复定义同一实体；同一个(name, id)始终指代同一概念。

4）“直接替代名词”规则（非常重要）：
- XML标签必须直接替代句子中的名词/名词短语，不能把名词保留再把标签跟在名词后面。
- 应输出“<entity name="肿瘤区" id="1">...</entity>边界清晰”，而不是“肿瘤区<entity name="肿瘤区" id="1">...</entity>边界清晰”
- 应输出“图中未出现<entity name="细胞核" id="2" status="neg"/>”，而不是“图中未出现细胞核<entity name="细胞核" id="2" status="neg"/>”
----------- 要求结束 -----------

为了防止产生错误依据，我会在元数据中将真实依据的bbox list提供给你，其为一个字典，key为视觉依据实体名，value为对应bbox list。
你仅需要按照要求生成思考和回答，然后将提供给你的真实bbox list插入到你的思考和回答中即可。
为了减轻负担，元数据中我未供原始bbox list，而是使用<bbox_list n="bbox数量">代替出现了的视觉实体证据，使用<no_bbox>代替未出现的实体。你在生成思考时仅需要根据情况插入对应占位符即可。
例如输出“<entity name="实体名" id="唯一整数ID"><bbox_list></entity>”，而不是“<entity name="实体名" id="唯一整数ID"><bbox_list><box>x1, y1, x2, y2</box></bbox_list></entity>”。
不过<no_bbox>仅代表实体未出现，因此生成时仍需按照要求输出阴性证据的格式：<entity name="实体名" id="唯一整数ID" status="neg"/>

要求：
- 上一个任务构造问答对时，一定要构造成回答时需要准确找出本部分的视觉依据才能正确回答的问题和选项
- 元数据中visual_evidence提供的视觉依据实体名，可以在构造思考回答时，根据当前语义和图像内容进行合适的修改，只要保证不偏离原意即可。
    + 例如实体名“tumor cell nucleus in vessel”，在实际推理时可以为“这个血管中有大量<entity name="癌细胞核"...</entity>”
    + 例如实体名“癌症区域”，在实际推理时可以为“其中有一个非常显眼、巨大的 <entity name="结节状病灶"...</entity>”
- <think></think>内遵循System 2式思考，且要同时基于图像内容和专业病理知识
- <think></think>内思考过程模拟实际病理科医生思考时内心所想，实体标签要随第一次发现或确认证据时给出，不要先定义再去图中找
- 首次提及实体证据必须在<think></think>内。后续引用可以在<think></think>内也可以在外面
- 如果构造的问题是选择题且尚未为答案分配选项序号，则<answer></answer>内直接存储正确答案的index
- <entity>标签的生成要自然替代相应文本：
    + 合适：这里的关键证据是脉管腔内<entity name=\"癌栓\" id=\"1\"><bbox_list></entity>的存在。
    + 不合适：这里的关键证据是脉管腔内癌栓的存在，即<entity name=\"癌栓\" id=\"1\"><bbox_list></entity>。
- 返回的json字典中添加reasoning字段存储本任务的构造结果

核心准则：
- 证据封闭原则： 证据封闭原则： 你定义的 <entity> 标签必须且仅限于元数据 visual_evidence 字典中提供的 key (但可以根据上面的要求的第2条修改表述)。禁止在 <entity> 标签中引入元数据之外的任何新实体名。
- 证据全覆盖原则： 元数据中给出的所有 visual_evidence 实体必须在 <think> 过程中全部被提及并标注。
- 返回的json字典中添加evidence_map字段，以字典形式存储visual_evidence中每个key对应的entity的id
- 返回的json字典中添加entity_map字段，以字典形式存储每个entity id对应的entity的visual_evidence中的key
""".strip()

# print(BASE_PROMPT.format(
#     meta_data="{'organ': 'liver', 'stain': 'H&E', 'diagnosis': 'MVI positive', 'visual_evidence': {'tumor cell cluster': '<bbox_list>'}}",
#     task1_requirement=DIAG_RELATED_QUENSTION_GENERATE_PROMPT,
#     task2_requirement=RSN_WITH_ENTITY_PROMPT))

In [ ]:
meta_data = load_json('/c22073/datasets/RSN_LOC/MVI/pos/train_metadata.json')
print(len(meta_data), '\n', meta_data[0])

In [ ]:
client = MultiModalLLM(provider="gemini")

In [ ]:
responses = [None] * len(meta_data)
for i, data in enumerate(tqdm(meta_data)):
    count = 0
    while True:
        res = client.chat_with_image(
            prompt=BASE_PROMPT.format(task1_requirement=DIAG_RELATED_QUENSTION_GENERATE_PROMPT, task2_requirement=RSN_WITH_ENTITY_PROMPT, meta_data=data['metadata']),
            image_path=data['image_path'],
        )
        if not res.startswith('Error with'):
            break
        count += 1
        if count <= 10:
            print(f"第{i}张图片{data['image_path']} 生成数据遇到错误，重试第{count}次")
        else:
            print(f"第{i}张图片{data['image_path']} 生成数据遇到错误，暂时跳过")
    responses[i] = res

In [ ]:
from json_repair import repair_json
res_json = [json.loads(repair_json(x)) for x in responses]
print(len(res_json))

In [ ]:
for i, data in enumerate(tqdm(res_json)):
    assert 'question' in data, i
    assert 'options' in data, i
    assert 'gt' in data, i
    assert 'visual_integration_challenges' in data, i
    assert 'reasoning' in data, i
    assert 'evidence_map' in data, i
    assert 'entity_map' in data, i
    assert isinstance(data['question'], str), i
    assert isinstance(data['options'], list), i
    assert isinstance(data['options'][0], str), i
    assert isinstance(data['gt'], int), i
    assert isinstance(data['reasoning'], str), i

In [ ]:
dump_json(res_json, '/c22073/datasets/RSN_LOC/MVI/pos/train_gemini_response_origin.json')

In [ ]:
def clean_neg_entity_tags(text):
    """
    只替换同时满足以下两个条件的标签：
    1. 含有 status="neg"
    2. 是自闭合标签 (以 /> 结尾)
    """
    # 正则表达式拆解：
    # 1. <entity\s+ : 匹配标签开始
    # 2. (?=[^>]*?status=\\?["']neg\\?["']) : 【关键】前瞻断言(Lookahead)。
    #    意思是：在此之后的标签内部，必须能找到 status="neg" 或 status=\"neg\"。
    #    如果找不到，整个匹配直接失败，跳过该标签。
    # 3. [^>]*? : 忽略 name 之前的其他属性
    # 4. name=\\?["']([^"']*?)\\?["'] : 捕获 name 的值
    # 5. [^>]*? : 忽略 name 之后的其他属性
    # 6. /> : 【关键】强制匹配自闭合结尾。如果标签以 > 结尾（非自闭合），匹配会失败。
    
    pattern = r'<entity\s+(?=[^>]*?status=\\?["\']neg\\?["\'])[^>]*?name=\\?["\']([^"\']*?)\\?["\'][^>]*?/>'
    
    # 替换逻辑：保留捕获组1（即 name 的内容）
    return re.sub(pattern, r'\1', text)

In [ ]:
# s = "<think>首先，观察这幅肝脏组织的HE染色图像。在视野中央，我注意到一个明显的空腔结构，其边缘衬有扁平的内皮细胞，这提示这是一个微型脉管（可能是门静脉小分支或肝静脉小分支）。接着，重点观察这个脉管的腔隙内部，发现其中并不是空的，也不是填充着红细胞或纤维素，而是存在一簇紧密排列的细胞。我仔细辨认这些细胞的形态，发现这些<entity name=\"肿瘤细胞核\" id=\"1\"><bbox_list></entity>具有明显的恶性特征：核体积显著增大，核浆比增高，且染色较深，核轮廓不规则。根据病理学诊断标准，在由内皮细胞包绕的脉管腔隙中出现恶性肿瘤细胞团，这是微血管侵犯（Microvascular Invasion, MVI）的典型组织学证据。图中未见明显的<entity name=\"纤维素血栓\" id=\"2\" status=\"neg\"/>，也未见单纯的炎症细胞堆积。因此，通过识别腔内异型的<ref name=\"肿瘤细胞核\" id=\"1\"/>及其所处的解剖位置，可以明确诊断为MVI阳性。\n\n</think>图像显示，在肝脏实质的微血管腔内出现了一团具有明显异型性的细胞。这些<ref name=\"肿瘤细胞核\" id=\"1\"/>表现为核大深染、聚集成簇，并被脉管内皮所包绕，符合微血管侵犯（MVI）的病理表现。这一发现是评估肝癌预后及制定后续治疗方案的重要指标。<answer>1</answer>"
# print(s)
# s = clean_neg_entity_tags(s)
# if '<think>' in s and '<think>\n' not in s:
#     s = s.replace('<think>', '<think>\n')
# if '</think>' in s and '\n</think>' not in s:
#     s = s.replace('</think>', '\n</think>')
# if '</think>' in s and '</think>\n' not in s:
#     s = s.replace('</think>', '</think>\n')
# print(s)
# s = clean_neg_entity_tags(s)
# if '<think>' in s and '<think>\n' not in s:
#     s = s.replace('<think>', '<think>\n')
# if '</think>' in s and '\n</think>' not in s:
#     s = s.replace('</think>', '\n</think>')
# if '</think>' in s and '</think>\n' not in s:
#     s = s.replace('</think>', '</think>\n')
# if '<answer>' in s and '\n<answer>' not in s:
#     s = s.replace('<answer>', '\n<answer>')
# print(s)

In [ ]:
for i, data in enumerate(tqdm(res_json)):
    s = clean_neg_entity_tags(data['reasoning'])
    if '<think>' in s and '<think>\n' not in s:
        s = s.replace('<think>', '<think>\n')
    if '</think>' in s and '\n</think>' not in s:
        s = s.replace('</think>', '\n</think>')
    if '</think>' in s and '</think>\n' not in s:
        s = s.replace('</think>', '</think>\n')
    if '<answer>' in s and '\n<answer>' not in s:
        s = s.replace('<answer>', '\n<answer>')
    data['reasoning'] = s

In [ ]:
dump_json(res_json, '/c22073/datasets/RSN_LOC/MVI/pos/train_gemini_response.json')

In [ ]:
# 查看并修改以后再加载
res_json = load_json('/c22073/datasets/RSN_LOC/MVI/pos/train_gemini_response.json')

In [ ]:
import random

def format_question(question, options, gt):
    # 创建索引列表并打乱顺序
    indices = list(range(len(options)))
    random.shuffle(indices)
    # 将选项按打乱后的顺序重新排列
    shuffled_options = [options[i] for i in indices]
    # 找到正确答案的新位置
    new_gt = indices.index(gt)
    # 生成选项字母（支持最多26个选项）
    option_letters = [chr(ord('A') + i) for i in range(len(options))]
    # 构建选项字符串
    options_str = ""
    for i, (letter, option) in enumerate(zip(option_letters, shuffled_options)):
        options_str += f"\n{letter}. {option}"
    # 拼接完整问题
    full_question = f"{question}{options_str}"
    # 正确答案选项
    answer = option_letters[new_gt]
    return full_question, answer

# 测试示例
# question_text = "下列哪一项描述的是肝血管瘤的典型病理特征？"
# options_list = [
#     "肝实质内可见大量的炎性细胞浸润",
#     "血管腔内存在肿瘤细胞团", 
#     "肝细胞呈现弥漫性的脂肪变性",
#     "汇管区可见明显的胆管增生",
#     '我不道啊'
# ]
# correct_index = 4
# formatted_q, answer = format_question(question_text, options_list, correct_index)
# print("问题：")
# print(formatted_q)
# print(f"\n正确答案：{answer}")

In [ ]:
## 构造最终数据集
SYSTEM_PROMPT = """
你是由浙江大学VIPA实验室开发的病理多模态智能助手，用于辅助病理医生进行专业、准确、高效的诊断。你能够根据用户输入的图片和文字指令或问题，给出相应的回答。

在给出回答之前，你**必须**先在心里进行思考。将思考内容放在<think>和</think>之间，在</think>之后进行正式回答。对于有确定性答案的问题，如选择题和填空题等，在回答的最后，将选项或答案放在<answer>和</answer>之间。例如：<think>思考内容</think>正式回答内容和分析<answer>选项或回答短语</answer>

你具备视觉定位（Visual Grounding）能力。在分析过程中，当你在图像中找到关键实体证据（如细胞、组织结构、区域）时，请遵守以下 XML 格式规范给出实体定位，并用 XML 标签直接替代原本的名词文本：

1. 首次提及可见实体（阳性），需要同时给出该实体的一个或多个检测框：
   使用 <entity name="实体名" id="唯一整数ID"><bbox_list><box>x1, y1, x2, y2</box></bbox_list></entity>
   
2. 首次提及未见实体（阴性/缺失证据：当“某实体未出现”本身是关键证据时），需要设置status为neg，不需要检测：
   使用自闭合标签 <entity name="实体名" id="唯一整数ID" status="neg"/>

3. 后续再次提及已定义的实体，需要准确引用之前提过的实体id，并保证name相同：
   使用自闭合标签 <ref name="实体名" id="对应ID"/>

注意：每个回答内实体 id 必须唯一，且为从1递增的整数。不要在标签前重复实体名称。例如，应给出“发现<entity name='细胞核' id='1'>...</entity>”，禁止给出“发现细胞核<entity name='细胞核' id='1'>...</entity>”。
""".strip()
random.seed(42)
mvi_chinses_train_dataset = []
for meta, gene in zip(meta_data, res_json):
    data = dict(images=[meta['image_path']])
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    
    q, a = format_question(gene['question'], gene['options'], gene['gt'])
    if random.random() < 0.5:
        question = '<image>\n' + q
    else:
        question = q + '\n<image>'
    messages.append({'role': 'user', 'content': question})
    
    reasoning = gene['reasoning']
    assert reasoning.count('<bbox_list>') == len(meta['metadata']['visual_evidence']), meta
    for v in meta['metadata']['visual_evidence'].values():
        reasoning = reasoning.replace('<bbox_list>', v, 1)
    assert f'<answer>{gene["gt"]}</answer>' in reasoning, meta
    messages.append({'role': 'assistant', 'content': reasoning.replace(f'<answer>{gene["gt"]}</answer>', f'<answer>{a}</answer>')})
    data['messages'] = messages
    mvi_chinses_train_dataset.append(data)
print(len(mvi_chinses_train_dataset))

In [ ]:
# 这里查看数据发现，有很多回答里按照原来的顺序充斥着选项1，选项A这种分析，需要手工修改。后续需要修正数据构造流程。
# 模型还会错误生成阴性实体的引用<ref name=\"肝细胞大泡性脂肪变性\" id=\"3\"/>，因此后续也需要修改
#dump_json(mvi_chinses_train_dataset, '/c22073/datasets/RSN_LOC/MVI/pos/train_convs_gemini.json')

## 1.2 WSI缩略图修改现有精细推理数据

In [4]:
# 获取原始推理数据集
ori_dataset = load_json('/c22073/codes/swift-new/projects/PathVerse/dataset/nips/1_sft/grpo_coldstart_all_rounds.json')
print(len(ori_dataset))
ori_dataset = [x for x in ori_dataset if 'tool' not in set([y['role'] for y in x['messages']]) and '可以调用的工具函数' not in x['messages'][0]['content']]
print(len(ori_dataset))
ori_dataset = [x for x in ori_dataset if 'patch' not in x['images'][0]]
print(len(ori_dataset))

252
184
68


In [5]:
prefix = '/c22073/datasets/combination/'
s = set()
for data in ori_dataset:
    assert len(data['images']) == 1, data
    img = data['images'][0]
    s.add(osp.dirname(img).replace(prefix, ''))
for x in sorted(list(s)):
    print(x)

HCC_grading/1
HCC_grading/2
HCC_grading/3
HCC_grading/4
ZheYi0607/ICC_subtype_thumbnails/train
ZheYi0607/liver_cancer_thumbnails/train


In [7]:
# 获取数据集中图像对应的癌症区域检测或分割标签
source_dataset = load_json('/c22073/public_datasets/pathology-vision-task/1_thumbnail_ChineseO.json') + load_json('/c22073/public_datasets/pathology-vision-task/1_thumbnail_English.json')

img2boxlist = {}
nobbox_images = []
for data in tqdm(source_dataset):
    responses = set([x['value'] for x in data['conversations'] if x['from'] == 'gpt'])
    bbox = None
    if 'No cancer detected.' in responses:
        bbox = '<no_bbox>'
    else:
        resbboxlist = [x for x in responses if '<bbox_list>' in x]
        assert len(resbboxlist) <= 1, data
        if len(resbboxlist) == 0:
            nobbox_images.append(osp.join(prefix, data['image']))
            continue
        bbox = resbboxlist[0]
        bbox = bboxes_to_str(str_to_bboxes(bbox))
    if bbox is not None:
        img2boxlist[osp.join(prefix, data['image'])] = bbox

new_dataset_raw = []
for data in tqdm(ori_dataset):
    assert len(data['images']) == 1, data
    img = data['images'][0]
    if img not in img2boxlist:
        if img not in nobbox_images:
            print(f"Can't find image {img}")
        else:
            print(f"Image {img} has no bbox")
        continue
    data = deepcopy(data)
    data['visual_evidence'] = img2boxlist[img]
    new_dataset_raw.append(data)
print(len(new_dataset_raw))

100%|██████████| 68/68 [00:00<00:00, 74604.41it/s]

Image /c22073/datasets/combination/ZheYi0607/liver_cancer_thumbnails/train/[cirrhosis]2023000110-5.jpg has no bbox
Image /c22073/datasets/combination/ZheYi0607/liver_cancer_thumbnails/train/[cirrhosis]2023009226-5.jpg has no bbox
Image /c22073/datasets/combination/ZheYi0607/ICC_subtype_thumbnails/train/[ICC (large bile duct type)]2023049923-10.jpg has no bbox
Image /c22073/datasets/combination/ZheYi0607/ICC_subtype_thumbnails/train/[ICC (small bile duct type)]2023013312-5.jpg has no bbox
Image /c22073/datasets/combination/ZheYi0607/liver_cancer_thumbnails/train/[cirrhosis]2023000110-5.jpg has no bbox
Image /c22073/datasets/combination/ZheYi0607/liver_cancer_thumbnails/train/[cirrhosis]2023009226-5.jpg has no bbox
Image /c22073/datasets/combination/ZheYi0607/ICC_subtype_thumbnails/train/[ICC (large bile duct type)]2023049923-10.jpg has no bbox
Image /c22073/datasets/combination/ZheYi0607/ICC_subtype_thumbnails/train/[ICC (small bile duct type)]2023013312-5.jpg has no bbox
60
